# Ejercicio: Web Scraping

### Nombre: Kenneth Yar

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

### Parte 0: Planificar
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

### Mi plan

- **Datos a obtener:** título, descripción, ingredientes, instrucciones y datos nutricionales de cada receta.
- **Sitio elegido:** allrecipes.com, arrancando desde la receta de Rotisserie Chicken guardada localmente.
- **Estructura del corpus:** parto de esa receta raíz, extraigo los links a otras recetas relacionadas que aparecen en la misma página, y repito la extracción para cada una. La meta es terminar con un corpus de recetas de pollo que después indexo con embeddings para poder hacerles preguntas con RAG.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

In [2]:
from bs4 import BeautifulSoup

file = 'rotisserie-chicken.html'

# Load the HTML file
with open(file, "r", encoding="utf-8") as file:
    html_content = file.read()
    
# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

In [3]:
# Extracting the recipe title
title = soup.find("meta", {"property": "og:title"})["content"]
title

'Rotisserie Chicken'

In [4]:
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
for ingredient in ingredients_section:
    print(ingredient.text.strip())

1 (3 pound) whole chicken
1 pinch salt
¼ cup butter, melted
1 tablespoon salt
1 tablespoon ground paprika
¼ tablespoon ground black pepper


## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [5]:
# Extracting the description
description = soup.find("meta", {"name": "description"})["content"]

# Extracting the ingredients
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

# Extracting the instructions
instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
instructions = [instruction.get_text().strip() for instruction in instructions_section]

# Extracting the nutrition information
nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

# Print the extracted information
print("Recipe Title:", title)
print("Description:", description)
print("Ingredients:")
for ingredient in ingredients:
    print("-", ingredient)
print("Instructions:")
for i, instruction in enumerate(instructions, 1):
    print(f"{i}. {instruction}")
print("Nutrition Facts:")
for fact in nutrition_facts:
    print("-", fact)


Recipe Title: Rotisserie Chicken
Description: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.
Ingredients:
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper
Instructions:
1. Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.
2. Here's what you'll need to make rotisserie chicken at home:
3. · Whole Chicken: This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time.· Butter: Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to.· Seasonings: The rotisserie chicken is simply seasoned with salt, pepper, and paprika.
4. You'll find the full, step-by-step recipe below — b

## Parte 3: Obtener enlaces relacionados
* Encontrar links a otras recetas para completar el corpus

Al revisar los links de la página con `soup.find_all("a", href=True)`, noté que un filtro amplio como `"recipe" in href` trae de todo: enlaces a categorías, menús de navegación, login, etc. Usar `/recipe/` como filtro ya es más preciso porque coincide con el patrón de URL real de una receta individual en este sitio (`/recipe/<numero>/<nombre>/`). 

Revisé el resultado impreso y en mi caso este filtro simple ya devolvió las 16 URLs limpias, sin duplicados ni links de navegación colados.

In [6]:
# Find all the links to other recipes
recipe_links = soup.find_all("a", href=True)

# Filter and print only the links that are likely to be recipes
recipe_urls = []
for link in recipe_links:
    href = link['href']
    if "/recipe/" in href:
        recipe_urls.append(href)

# Print the recipe URLs
print("Linked Recipes:")
for url in recipe_urls:
    print(url)

Linked Recipes:
https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/
https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/
https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken/
https://www.allrecipes.com/recipe/14531/beer-butt-chicken/
https://www.allrecipes.com/recipe/221093/good-frickin-paprika-chicken/
https://www.allrecipes.com/recipe/264278/miso-honey-chicken/
https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken/
https://www.allrecipes.com/recipe/222936/smoked-beer-butt-chicken/
https://www.allrecipes.com/recipe/228070/the-best-beer-can-chicken-ever/
https://www.allrecipes.com/recipe/214619/bbq-beer-can-chicken/
https://www.allrecipes.com/recipe/19944/drunk-chicken/
https://www.allrecipes.com/recipe/275044/grilled-chicken-under-a-brick/
https://www.allrecipes.com/recipe/281255/smoked-whole-chicken/
https://www.allrecipes.com/recipe/34957/easy-barbeque-chicken/
https://www.allrecipes.com/recipe/8998/darn-good-chick

In [ ]:
def extraer_receta(soup, url=None):
    # Si la página no tiene el meta tag og:title, evitamos que truene el programa
    # y devolvemos un valor por defecto en vez de un error de tipo NoneType
    title_tag = soup.find("meta", {"property": "og:title"})
    title = title_tag["content"] if title_tag else "Sin título"

    desc_tag = soup.find("meta", {"name": "description"})
    description = desc_tag["content"] if desc_tag else ""

    # Ingredientes: cada uno vive en un <li> con esta clase específica
    ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
    ingredients = [i.get_text().strip() for i in ingredients_section]

    # Instrucciones: párrafos del cuerpo de la receta
    instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
    instructions = [i.get_text().strip() for i in instructions_section]

    # Nutrición: el nombre del nutriente y su valor comparten el mismo <li> padre,
    # por eso usamos .parent para agarrar ambos textos juntos
    nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
    nutrition_facts = [n.parent.get_text().strip().replace('\n', ' ') for n in nutrition_section]

    return {
        "url": url,
        "title": title,
        "description": description,
        "ingredients": ingredients,
        "instructions": instructions,
        "nutrition_facts": nutrition_facts
    }

In [10]:
receta_raiz = extraer_receta(soup, url="rotisserie-chicken.html")
receta_raiz

{'url': 'rotisserie-chicken.html',
 'title': 'Rotisserie Chicken',
 'description': "Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.",
 'ingredients': ['1 (3 pound) whole chicken',
  '1 pinch salt',
  '¼ cup butter, melted',
  '1 tablespoon salt',
  '1 tablespoon ground paprika',
  '¼ tablespoon ground black pepper'],
 'instructions': ["Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.",
  "Here's what you'll need to make rotisserie chicken at home:",
  "· Whole Chicken: This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time.· Butter: Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to.· Seasonings: The rotisserie chicken is simply seasoned with salt, pepper, 

Para descargar las 16 recetas decidí no usar solo un `time.sleep()` fijo entre peticiones, sino agregar manejo de **timeouts con reintentos** (hasta un número limitado de intentos por URL) y una **pausa aleatoria** entre 3 y 6 segundos en vez de un valor fijo. La razón es que una petición puede fallar por una lentitud momentánea de la red o del servidor sin que eso signifique que el sitio me está bloqueando — un reintento evita perder esa receta por un problema pasajero. La pausa aleatoria, en vez de fija, hace el patrón de tráfico menos predecible y más parecido al de un usuario real navegando, lo cual es más respetuoso con el servidor.

Antes de hacer las peticiones automatizadas, agregué un `User-Agent` realista y pausas entre peticiones para no sobrecargar el servidor ni comportarme como un bot lo que son buenas prácticas básicas de cualquier scraper, independientemente de si el sitio bloquea o no explícitamente este tipo de tráfico.

In [ ]:
import random
import requests
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9,es;q=0.8",
    "Connection": "keep-alive"
}

corpus = []
corpus.append(receta_raiz)

for i, url in enumerate(recipe_urls, 1):
    intentos = 3
    exito = False
    
    while intentos > 0 and not exito:
        try:
            # Subimos el timeout a 20 para dar margen al servidor
            resp = requests.get(url, headers=headers, timeout=20)
            resp.raise_for_status()
            
            sp = BeautifulSoup(resp.text, "html.parser")
            receta = extraer_receta(sp, url=url)
            corpus.append(receta)
            
            print(f"[{i}] OK: {receta.get('title', 'Sin título')}")
            exito = True  # Rompe el bucle 'while' si todo sale bien
            
        except requests.exceptions.Timeout:
            intentos -= 1
            if intentos > 0:
                print(f"[{i}] TIMEOUT en {url}. Reintentando... (Quedan {intentos} intentos)")
                time.sleep(5)  # Espera 5 segundos antes de reintentar
            else:
                print(f"[{i}] ERROR definitivo por TIMEOUT en {url}")
                
        except Exception as e:
            print(f"[{i}] ERROR crítico con {url}: {e}")
            break

    # Pausa ALEATORIA entre 3 y 6 segundos
    pausa = random.uniform(3.0, 6.0)
    time.sleep(pausa)

print(f"\nTotal recetas en corpus: {len(corpus)}")

[1] TIMEOUT en https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/. Reintentando... (Quedan 2 intentos)
[1] OK: Cilantro-Lime Grilled Chicken
[2] TIMEOUT en https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/. Reintentando... (Quedan 2 intentos)
[2] OK: Buttermilk Barbecue Chicken
[3] OK: Grilled Spatchcocked Chicken
[4] OK: Beer Butt Chicken
[5] OK: Good Frickin’ Paprika Chicken
[6] OK: Miso Honey Chicken
[7] OK: Rosemary Buttermilk Chicken
[8] OK: Smoked Beer Butt Chicken
[9] OK: The Best Beer Can Chicken Ever
[10] OK: Best Beer Can Chicken
[11] OK: Drunk Chicken
[12] OK: Grilled Chicken Under a Brick
[13] OK: Smoked Whole Chicken
[14] OK: Easy Barbeque Chicken
[15] TIMEOUT en https://www.allrecipes.com/recipe/8998/darn-good-chicken/. Reintentando... (Quedan 2 intentos)
[15] OK: Darn Good Chicken
[16] OK: Beer Can Chicken

Total recetas en corpus: 17


## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

In [18]:
documentos = []
for receta in corpus:
    texto = f"Receta: {receta['title']}\n"
    texto += f"Descripcion: {receta['description']}\n"
    texto += f"Ingredientes: {', '.join(receta['ingredients'])}\n"
    texto += f"Instrucciones: {' '.join(receta['instructions'])}"
    documentos.append(texto)

print(f"Documentos: {len(documentos)}")
print("---")
print(documentos[0][:500])

Documentos: 17
---
Receta: Rotisserie Chicken
Descripcion: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.
Ingredientes: 1 (3 pound) whole chicken, 1 pinch salt, ¼ cup butter, melted, 1 tablespoon salt, 1 tablespoon ground paprika, ¼ tablespoon ground black pepper
Instrucciones: Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'l


In [19]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(documentos, show_progress_bar=True)

print(f"Shape: {embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Shape: (17, 384)


In [20]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def buscar(query, top_k=3):
    query_emb = model.encode([query])
    sims = cosine_similarity(query_emb, embeddings)[0]
    indices = np.argsort(sims)[::-1][:top_k]

    resultados = []
    for idx in indices:
        resultados.append({
            "title": corpus[idx]["title"],
            "score": sims[idx],
            "texto": documentos[idx]
        })
    return resultados

In [21]:
resultados = buscar("chicken with beer")
for r in resultados:
    print(f"{r['score']:.4f} - {r['title']}")

0.5918 - Drunk Chicken
0.5678 - Beer Can Chicken
0.5671 - Beer Butt Chicken


In [35]:
from google import genai

# La API key se lee desde un archivo de texto local, NUNCA escrita directamente en el código
with open("api_key.txt", "r") as f:
    api_key = f.read().strip()

client = genai.Client(api_key=api_key)

model_name = "gemini-3.1-flash-lite"


In [36]:
def rag(query, top_k=3):
    resultados = buscar(query, top_k)

    contexto = ""
    for r in resultados:
        contexto += f"--- {r['title']} (score: {r['score']:.4f}) ---\n"
        contexto += r["texto"] + "\n\n"

    prompt = f"""Basandote SOLO en las siguientes recetas, responde la pregunta del usuario.
Responde siempre en español.

{contexto}

Pregunta: {query}
Respuesta:"""

    respuesta = client.models.generate_content(
        model=model_name,
        contents=prompt
    )
    return respuesta.text, resultados

In [40]:
respuesta, docs = rag("How do I make chicken with beer?")
print("Documentos usados:")
for d in docs:
    print(f"  {d['score']:.4f} - {d['title']}")
print(f"\nRespuesta:\n{respuesta}")

Documentos usados:
  0.7256 - Beer Can Chicken
  0.7224 - Drunk Chicken
  0.7190 - Beer Butt Chicken

Respuesta:
Puedes preparar pollo con cerveza siguiendo cualquiera de estas tres recetas proporcionadas:

**1. Beer Can Chicken:**
*   **Condimentos:** Mezcla azúcar morena, chile en polvo, pimentón (paprika), mostaza seca, sal y pimienta negra.
*   **Preparación:** Coloca un pollo entero sobre media lata de cerveza. Esparce una cucharadita de mezcla de especias dentro de la cavidad y frota el resto sobre la piel.
*   **Cocción:** Cocina a fuego indirecto (375°F / 190°C) durante aproximadamente 1 hora y 15 minutos, hasta que un termómetro indique 165°F (74°C). Deja reposar 10 minutos antes de servir.

**2. Drunk Chicken:**
*   **Condimentos:** Usa sazonador para aves, humo líquido y hojas de laurel.
*   **Preparación:** Coloca las hojas de laurel bajo la piel y cubre el pollo con el sazonador. Bebe la mitad de la cerveza, vierte el humo líquido en la lata e insértala en el pollo. Usa un

In [45]:
respuesta2, docs2 = rag("Quiero hacer pollo a la parrilla fácil")
print("Documentos usados:")
for d in docs2:
    print(f"  {d['score']:.4f} - {d['title']}")
print(f"\nRespuesta:\n{respuesta2}")

Documentos usados:
  0.2214 - Cilantro-Lime Grilled Chicken
  0.2125 - Easy Barbeque Chicken
  0.1848 - Rosemary Buttermilk Chicken

Respuesta:
Para hacer pollo a la parrilla de forma fácil, puedes optar por la receta **Easy Barbeque Chicken**, la cual utiliza una marinada sencilla de jugo de limón, aceite vegetal, vinagre, orégano seco y ajo en polvo.

Los pasos básicos son:
1. Mezclar los ingredientes de la marinada en un bol grande.
2. Colocar las piezas de pollo, sazonar con sal y pimienta, y dejar marinar en el refrigerador al menos 1 hora.
3. Precalentar la parrilla a fuego alto y aceitar ligeramente la rejilla.
4. Cocinar el pollo hasta que los jugos sean claros, pincelándolo ocasionalmente con la marinada sobrante durante la cocción. (Recuerda desechar el resto de la marinada al finalizar).


### Prueba de fidelidad al contexto

Una de las ventajas de RAG frente a solo preguntarle al LLM directamente es que la respuesta está anclada en el corpus que le paso como contexto. Para comprobar que mi implementación realmente respeta eso y no "alucina" información que no está en las recetas, le hago una pregunta que no tiene relación con ninguna receta de pollo (sushi). Si el sistema funciona bien, el modelo debería reconocer que no tiene esa información en vez de inventarse una receta.

In [ ]:
respuesta3, docs3 = rag("How do I make sushi?")
print(f"Respuesta:\n{respuesta3}")

Respuesta:
Lo siento, pero ninguna de las recetas proporcionadas contiene instrucciones para hacer sushi.


In [42]:
print("Resumen del corpus")
print(f"Total recetas: {len(corpus)}")
total_ings = sum(len(r['ingredients']) for r in corpus)
print(f"Ingredientes totales: {total_ings} (promedio {total_ings/len(corpus):.1f} por receta)")
total_pasos = sum(len(r['instructions']) for r in corpus)
print(f"Pasos totales: {total_pasos} (promedio {total_pasos/len(corpus):.1f} por receta)")
print(f"Dimensión embeddings: {embeddings.shape}")

Resumen del corpus
Total recetas: 17
Ingredientes totales: 137 (promedio 8.1 por receta)
Pasos totales: 118 (promedio 6.9 por receta)
Dimensión embeddings: (17, 384)


## Conclusión

En este ejercicio a partir de una sola receta guardada en HTML local, extraí sus datos con BeautifulSoup, indentigiqué los links a recetas relacionadas y construí un pequeño crawler (con manejo de errores por timeout y pausas entre peticiones) para descargar y procesar 16 recetas más. Con el corpus de 17 recetas generé embeddings usando `all-MiniLM-L6-v2` y armé una función de búsqueda por similitud coseno. Finalmente combiné esa función de retrieval junto con Gemini para implementar RAG: las respuestas generadas están fundamentadas en el corpus real, y lo confirmé probando con una pregunta fuera de dominio (sushi), donde el modelo reconoció correctamente que no tenía esa información en vez de inventarla.